# Artificial Intelligence in Health Care

# NB1: Getting Started
### Your environment, and a neural network computed by hand

This notebook has two goals:

1. **Check that your environment works**
2. **Warm-up:** Compute a neural network by hand, and then get Keras to produce exactly
   the same number.

Nothing here learns anything. There is no training, no optimizer, no loss
function. That is deliberate: by the end you will have *run* a network, which
leaves the question: *How does it find those weights?* 

> **Have you set up the course environment yet?** If not, do the section
> immediately below first. If you already have the `aihc` environment running,
> skip straight to §0.

---
## Before you start: setting up your environment

You do this **once**. You need two files from Moodle, `environment.yml` and
`requirements.txt`; put them in a folder and open a terminal (Anaconda Prompt; as Admin) there.
Note: conda env is recommended here. You could also take care of your python environment by yourself.

### Anaconda or Miniconda [recommended]

```bash
conda env create -f environment.yml
conda activate aihc
pip install -r requirements.txt
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
```

**On Windows, run these in the Anaconda Prompt**, not in PowerShell or `cmd` —
unless you have previously run `conda init`, those shells will not find `conda`.

Note what `environment.yml` actually installs: **Python 3.11 and pip, and
nothing else.** Everything on top comes from pip. PyTorch publishes its CPU
builds on its own package index rather than to conda.

**Your other environments are untouched.** If you have conda environments for
other courses, `aihc` sits beside them and changes nothing.

### Then: start VS Code (or others) and select the right kernel

Open this notebook, and **look at the kernel name in the top-right corner**. It
must read **`aihc`**. If it says anything else, click it and pick from the list.

> This one step is the single most common reason §0 fails with
> `ModuleNotFoundError` on a machine where the installation
> worked. The packages went into the `aihc` environment; the notebook is running
> somewhere else.

---
## 0 Does your environment work?

Run the next cell. If it prints a version table and the words at the bottom, you
are ready!

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"    # must be set BEFORE keras is imported

import sys, platform
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import torch
import keras

rows = [
    ("Python",       platform.python_version()),
    ("numpy",        np.__version__),
    ("pandas",       pd.__version__),
    ("scikit-learn", sklearn.__version__),
    ("matplotlib",   matplotlib.__version__),
    ("torch",        torch.__version__),
    ("keras",        keras.__version__),
    ("keras backend", keras.backend.backend()),
]
width = max(len(a) for a, _ in rows)
for a, b in rows:
    print(f"{a:<{width}}   {b}")

assert keras.backend.backend() == "torch", "KERAS_BACKEND is not set to torch"
print()
print("Environment OK - you are ready for Session 2.")

Python          3.11.15
numpy           2.4.6
pandas          3.0.5
scikit-learn    1.9.0
matplotlib      3.11.1
torch           2.13.0+cpu
keras           3.15.1
keras backend   torch

Environment OK - you are ready for Session 2.


### Why `KERAS_BACKEND = "torch"`

You will write **Keras**. Keras is a high-level interface: underneath it, some library does
the actual arithmetic. In this course that library is **PyTorch**.

You will not interact with PyTorch directly except once, when we
look at how gradients are computed. The line above simply tells Keras which
engine to use, and it has to run *before* `import keras`.

---
## 1 The dataset

**UCI Heart Disease** (Cleveland). 303 patients, 13 clinical features, and a
diagnosis. You know it already!

The raw target `num` runs from 0 to 4 (degree of vessel narrowing). We binarise
it the way the literature on this dataset conventionally does:
**`num > 0` means disease present.**

In [ ]:
from ucimlrepo import fetch_ucirepo

heart = fetch_ucirepo(id=45)
X_all = heart.data.features
y = (heart.data.targets["num"] > 0).astype(int).values

print("patients :", X_all.shape[0])
print("features :", X_all.shape[1])
print("positive rate: %.1f%%" % (100 * y.mean()))
print()
print(list(X_all.columns))

patients : 303
features : 13
positive rate: 45.9%

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


Two things to notice, because they matter later:

- **One row per patient.** There is no patient identifier and no repeated
  records. That makes this dataset unusually simple which is *not* typical.
  You often deal with data where the same
  patient appears many times. This needs to be considered when doing data splitting.
- **The classes are almost balanced** (about 46 % positive). Also not typical of
  clinical prediction, where we typically have unbalanced classes.

---
## 2 One patient, three features

We skip exploratory data analysis here. We just need exactly three numbers.

| Feature | Meaning |
|---|---|
| `age` | age in years |
| `trestbps` | resting blood pressure (mm Hg) |
| `thalach` | maximum heart rate achieved during exercise testing |

In [ ]:
FEATURES = ["age", "trestbps", "thalach"]

# Standardise (w/o sklearn): a neuron's weights are only interpretable if its inputs are on a
# comparable scale. With raw values (age ~54, blood pressure ~131) you would need
# very different weight magnitudes to say the same thing.
mu = X_all[FEATURES].mean()
sd = X_all[FEATURES].std()
Xs = ((X_all[FEATURES] - mu) / sd).values

i = 7                                        # an arbitrary patient
x = Xs[i]

print("raw values      :", X_all[FEATURES].iloc[i].to_dict())
print("standardised    :", np.round(x, 3))
print("actual diagnosis:", "disease" if y[i] else "no disease")

raw values      : {'age': 57, 'trestbps': 120, 'thalach': 163}
standardised    : [ 0.283 -0.664  0.585]
actual diagnosis: no disease


---
## 3 A neuron, by hand

A single neuron does exactly two things:

$$z \;=\; w_1x_1 + w_2x_2 + w_3x_3 + b \;=\; \mathbf{w}\cdot\mathbf{x} + b$$

$$p \;=\; \sigma(z) \;=\; \frac{1}{1 + e^{-z}}$$

The first line is a weighted sum (a dot product plus an offset/bias term). The second
maps any real number into $(0, 1)$ so it can be read as a probability.

That is the whole thing. **A neural network is this, repeated and stacked.**

We will choose the weights ourselves. Let us encode a crude clinical intuition:
older age raises risk, higher resting blood pressure raises risk slightly, and a
higher maximum heart rate lowers it.

In [ ]:
w = np.array([0.8, 0.3, -0.9])     # age, trestbps, thalach
b = -0.2

z = float(np.dot(w, x) + b)        # step 1: weighted sum
p = 1 / (1 + np.exp(-z))           # step 2: sigmoid

print("weighted sum  z = %.4f" % z)
print("probability   p = %.4f" % p)
print()
print("This patient:", "disease" if y[i] else "no disease")

weighted sum  z = -0.6995
probability   p = 0.3319

This patient: no disease


### What do you expect?

Before running the next cell: we are going to **increase the weight on `age`**
from 0.8 to 1.5, leaving everything else alone.

This patient's standardised age is printed above. Will the output probability go
**up** or **down**?

Decide, then run.

In [ ]:
for w_age in [0.3, 0.8, 1.5]:
    w_try = np.array([w_age, 0.3, -0.9])
    z_try = float(np.dot(w_try, x) + b)
    print(f"w_age = {w_age:>4}   ->   z = {z_try:>7.4f}   p = {1/(1+np.exp(-z_try)):.4f}")

print()
print("standardised age of this patient: %.3f" % x[0])

w_age =  0.3   ->   z = -0.8412   p = 0.3013
w_age =  0.8   ->   z = -0.6995   p = 0.3319
w_age =  1.5   ->   z = -0.5012   p = 0.3773

standardised age of this patient: 0.283


The direction follows the **sign of the patient's feature value**, not the sign
of the weight alone. This patient is slightly older than average (a positive
standardised age) so putting more weight on age pushes the prediction up. For a
younger-than-average patient the very same change would push it *down*.


---
## 4 The same neuron, in Keras

Now we build a Keras model with **one** `Dense` unit, force it to use the weights
we just chose by hand, and check what it outputs.

`Dense(1)` with no activation computes exactly $\mathbf{w}\cdot\mathbf{x} + b$ the weighted sum, and nothing else. It returns $z$, not $p$.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(1),            # no activation -> returns z (see later when we add a sigmoid activation)
])

# Keras wants the kernel shaped (n_inputs, n_units) and the bias shaped (n_units,)
model.set_weights([w.reshape(3, 1), np.array([b])])

z_keras = float(model.predict(x.reshape(1, 3), verbose=0)[0, 0])

print("z, by hand  : %.6f" % z)
print("z, by Keras : %.6f" % z_keras)
print("difference  : %.2e" % abs(z - z_keras))

z, by hand  : -0.699512
z, by Keras : -0.699512
difference  : 3.58e-08


Identical to machine precision --> because it is the same arithmetic.

### Why the model returns $z$ and not $p$

`Dense(1)` deliberately has no `activation="sigmoid"`. The raw output $z$ is
called a **logit**. To read a probability you apply the sigmoid yourself:

In [ ]:
p_keras = 1 / (1 + np.exp(-z_keras))
print("probability from the Keras model: %.4f" % p_keras)
print("probability computed by hand    : %.4f" % p)

probability from the Keras model: 0.3319
probability computed by hand    : 0.3319


**You have just run a neural network with predefined weights (forward propagation). But how does it learn?**